# DeepLabV3+ v5 — 2.5D Multi-View Sismik Fasiyes Segmentasyonu
**Veri Seti:** F3 Block Netherlands — Alaudah et al. (2019)  
**Model:** DeepLabV3+ (ResNet-50, in_channels=3, ImageNet pretrained)  
**Ortam:** CUDA / MPS (Apple Silicon) / CPU  

### v3 → v5 Temel İyileştirmeler
| # | Değişiklik | Neden |
|---|---|---|
| 1 | **2.5D input** (3 komşu slice → 3ch) | Komşu slice sürekliliği, ImageNet uyumu |
| 2 | **Inline + Crossline birlikte eğitim** | Test2 domain shift çözümü |
| 3 | **VerticalFlip kaldırıldı** | Derinlik ekseni ters çevrilmemeli |
| 4 | **WeightedRandomSampler** | Nadir sınıfları daha sık örnekle |
| 5 | **Blok-bazlı val split** | Spatial leakage'ı önle |
| 6 | **Focal Loss (γ=2)** | Azınlık sınıflara odaklan |
| 7 | **AdamW + OneCycleLR** | Daha güçlü regularizasyon |
| 8 | **Gradient Accumulation (×4)** | Efektif batch=32 |
| 9 | **TTA** (HFlip + Polarity) | Bedava +1-3% mIoU |

## 0. Kurulum ve Cihaz

In [ ]:
import subprocess, sys
pkgs = [
    "matplotlib",
    "scikit-learn",
    "albumentations",
    "segmentation-models-pytorch",
]
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet'] + pkgs)
print("Kurulum tamamlandi.")

In [ ]:
import os, sys, random, json, time, zipfile
from pathlib import Path
from urllib.request import urlretrieve

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, ConcatDataset, WeightedRandomSampler
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.patches as mpatches
from sklearn.metrics import confusion_matrix
import segmentation_models_pytorch as smp
import albumentations as A

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

if torch.cuda.is_available():
    device = torch.device("cuda")
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
elif hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
    device = torch.device("mps")
    print("Apple Silicon MPS backend aktif")
else:
    device = torch.device("cpu")
    print("UYARI: GPU bulunamadi, CPU kullaniliyor!")
print(f"Cihaz: {device} | PyTorch: {torch.__version__} | SMP: {smp.__version__}")

NOTEBOOK_DIR = Path(".").resolve()
PROJECT_DIR  = NOTEBOOK_DIR
DATA_DIR     = PROJECT_DIR / "data"
CHECKPOINTS  = PROJECT_DIR / "checkpoints_v5"
FIGURES_DIR  = PROJECT_DIR / "results_v5" / "figures"
METRICS_DIR  = PROJECT_DIR / "results_v5" / "metrics"

for d in [DATA_DIR, CHECKPOINTS, FIGURES_DIR, METRICS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print(f"Proje : {PROJECT_DIR}")
print(f"Veri  : {DATA_DIR}")
print(f"  train_seismic.npy mevcut: {(DATA_DIR / 'train' / 'train_seismic.npy').exists()}")

## 1. Veri Yükleme

In [ ]:
TRAIN_SEISMIC = DATA_DIR / "train" / "train_seismic.npy"

if not TRAIN_SEISMIC.exists():
    zip_path = DATA_DIR / "data.zip"
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    print("Veri indiriliyor (~1 GB)...")

    def _progress(block, block_size, total):
        downloaded = block * block_size
        if total > 0:
            pct = min(100, downloaded * 100 / total)
            print(f"\r  {pct:.1f}%  ({downloaded/1e6:.0f} MB)", end="", flush=True)

    urlretrieve("https://zenodo.org/record/3755060/files/data.zip",
                zip_path, reporthook=_progress)
    print("\nCikartiliyor...")
    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(DATA_DIR)
    zip_path.unlink()
    print("Hazir.")
else:
    print("Veri zaten mevcut.")

print("NPY yukleniyor...")
train_seismic = np.load(DATA_DIR / "train"     / "train_seismic.npy")
train_labels  = np.load(DATA_DIR / "train"     / "train_labels.npy")
test1_seismic = np.load(DATA_DIR / "test_once" / "test1_seismic.npy")
test1_labels  = np.load(DATA_DIR / "test_once" / "test1_labels.npy")
test2_seismic = np.load(DATA_DIR / "test_once" / "test2_seismic.npy")
test2_labels  = np.load(DATA_DIR / "test_once" / "test2_labels.npy")

print(f"Train volume : {train_seismic.shape}  dtype={train_seismic.dtype}")
print(f"  -> Inline sayisi  : {train_seismic.shape[0]}")
print(f"  -> Crossline sayisi: {train_seismic.shape[1]}")
print(f"  -> Derinlik       : {train_seismic.shape[2]}")
print(f"Test1 (inline)    : {test1_seismic.shape}")
print(f"Test2 (crossline) : {test2_seismic.shape}")

## 2. EDA

In [ ]:
CLASS_NAMES   = ["Upper NS", "Lower NS", "Rijnland", "Scruff", "Zechstein", "Under Zech"]
NUM_CLASSES   = 6
FACIES_COLORS = ["#3288bd", "#66c2a5", "#abdda4", "#e6f598", "#fdae61", "#f46d43"]
cmap_facies   = mcolors.ListedColormap(FACIES_COLORS)

counts = np.bincount(train_labels.flatten(), minlength=NUM_CLASSES)
total  = counts.sum()

print("Sinif dagilimi (Train volume):")
for i, (name, cnt) in enumerate(zip(CLASS_NAMES, counts)):
    print(f"  S{i} {name:22s}: {cnt:>12,}  ({cnt/total*100:.2f}%)")

fig, axes = plt.subplots(2, 2, figsize=(14, 8))

# Sinif dagilimi
bars = axes[0,0].bar(range(NUM_CLASSES), counts / 1e6, color=FACIES_COLORS, edgecolor="k", lw=0.5)
axes[0,0].set_xticks(range(NUM_CLASSES))
axes[0,0].set_xticklabels([f"S{j}\n{CLASS_NAMES[j]}" for j in range(NUM_CLASSES)], fontsize=8)
axes[0,0].set_ylabel("Piksel (Milyon)")
axes[0,0].set_title("Sinif Dagilimi")

# Inline ornek
idx_inl = train_seismic.shape[0] // 2
axes[0,1].imshow(train_seismic[idx_inl].T, cmap="seismic", aspect="auto", vmin=-1, vmax=1)
axes[0,1].set_title(f"Inline #{idx_inl}")

# Crossline ornek
idx_xln = train_seismic.shape[1] // 2
axes[1,0].imshow(train_seismic[:, idx_xln, :].T, cmap="seismic", aspect="auto", vmin=-1, vmax=1)
axes[1,0].set_title(f"Crossline #{idx_xln}")

# Inline label
axes[1,1].imshow(train_labels[idx_inl].T, cmap=cmap_facies, vmin=-0.5, vmax=5.5, aspect="auto")
axes[1,1].set_title(f"Fasiyes — Inline #{idx_inl}")

plt.suptitle("F3 Block — Inline ve Crossline Kesitler", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "eda_multiview.png", dpi=150, bbox_inches="tight")
plt.show()

## 3. Blok-Bazlı Train/Val Split + Sınıf Ağırlıkları
Son %20 inline'ları val olarak ayırıyoruz — spatial olarak birbirinden ayrık.

In [ ]:
n_inlines   = train_seismic.shape[0]
n_crosslines = train_seismic.shape[1]

# Blok-bazli split: ilk %80 train, son %20 val (spatial olarak ayrik)
val_ratio  = 0.2
val_start  = int(n_inlines * (1 - val_ratio))

train_inline_idx = np.arange(0, val_start)
val_inline_idx   = np.arange(val_start, n_inlines)
train_xline_idx  = np.arange(0, n_crosslines)  # tum crossline'lar train'de

print(f"Train inline : [0, {val_start})  -> {len(train_inline_idx)} slice")
print(f"Val inline   : [{val_start}, {n_inlines})  -> {len(val_inline_idx)} slice")
print(f"Train xline  : [0, {n_crosslines})  -> {len(train_xline_idx)} slice")
print(f"Toplam train : {len(train_inline_idx) + len(train_xline_idx)} slice")
print(f"\nNot: Crossline slice'lar val inline bolgesinden de gecer (minor spatial leak),")
print(f"     ama test verisi tamamen ayri volume oldugu icin gercek degerlendirme etkilenmez.")

# Sinif agirliklari — full train volume'den hesapla
train_counts  = np.bincount(train_labels[:val_start].flatten(), minlength=NUM_CLASSES).astype(float)
class_freq    = train_counts / train_counts.sum()
focal_alpha   = 1.0 / (class_freq + 1e-8)
focal_alpha   = focal_alpha / focal_alpha.sum()
focal_alpha_t = torch.FloatTensor(focal_alpha).to(device)

print("\nSinif frekanslari ve Focal alpha:")
for i, (name, f, a) in enumerate(zip(CLASS_NAMES, class_freq, focal_alpha)):
    print(f"  S{i} {name:22s}: freq={f:.4f}  alpha={a:.4f}")

## 4. 2.5D Dataset + Multi-View DataLoader + Rare-Class Sampling

In [ ]:
IMG_SIZE    = (256, 256)
BATCH_SIZE  = 8
ACCUM_STEPS = 4  # efektif batch = 32

# Augmentasyon — VerticalFlip YOK (derinlik ekseni ters cevrilmemeli)
train_transform = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.ShiftScaleRotate(shift_limit=0.1, scale_limit=0.15, rotate_limit=10,
                       border_mode=0, p=0.5),
    A.ElasticTransform(alpha=80, sigma=10, p=0.3),
    A.GridDistortion(num_steps=5, distort_limit=0.2, p=0.3),
    A.RandomBrightnessContrast(brightness_limit=0.15, contrast_limit=0.15, p=0.4),
    A.GaussNoise(std_range=(0.001, 0.015), p=0.3),
    A.CoarseDropout(max_holes=6, max_height=20, max_width=20, fill=0, p=0.2),
])


class F3Dataset25D(Dataset):
    """
    2.5D Dataset: 3 komsu slice -> 3 kanalli input.
    axis=0 -> inline slicing,  axis=1 -> crossline slicing.
    """
    def __init__(self, volume, labels, indices, axis=0, img_size=IMG_SIZE,
                 transform=None, augment_polarity=False):
        self.volume   = volume
        self.labels   = labels
        self.indices  = indices
        self.axis     = axis
        self.n_slices = volume.shape[axis]
        self.img_size = img_size
        self.transform = transform
        self.augment_polarity = augment_polarity

    def _get_slice(self, vol, idx):
        if self.axis == 0:
            return vol[idx]
        else:
            return vol[:, idx]

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        i = self.indices[idx]
        i_prev = max(0, i - 1)
        i_next = min(self.n_slices - 1, i + 1)

        # 2.5D: 3 komsu slice -> (H, W, 3) albumentations formatinda
        s_prev = self._get_slice(self.volume, i_prev).astype(np.float32)
        s_curr = self._get_slice(self.volume, i).astype(np.float32)
        s_next = self._get_slice(self.volume, i_next).astype(np.float32)
        img  = np.stack([s_prev, s_curr, s_next], axis=-1)  # (H, W, 3)
        mask = self._get_slice(self.labels, i).astype(np.int64)

        if self.augment_polarity and random.random() > 0.5:
            img = -img

        if self.transform is not None:
            aug  = self.transform(image=img, mask=mask.astype(np.uint8))
            img  = aug["image"]
            mask = aug["mask"].astype(np.int64)

        # (H, W, 3) -> (3, H, W)
        img_t  = torch.from_numpy(img.copy()).permute(2, 0, 1).float()
        mask_t = torch.from_numpy(mask.copy())

        img_t  = F.interpolate(img_t.unsqueeze(0), size=self.img_size,
                               mode="bilinear", align_corners=False).squeeze(0)
        mask_t = F.interpolate(mask_t.float().unsqueeze(0).unsqueeze(0),
                               size=self.img_size, mode="nearest").squeeze(0).squeeze(0).long()
        return img_t, mask_t


# --- Volume'u bir kez normalize et ---
train_mean = train_seismic.mean()
train_std  = train_seismic.std() + 1e-8
train_seis_norm = ((train_seismic - train_mean) / train_std).astype(np.float32)

# Test verisi de train stats ile normalize edilir (data leakage onlemi)
test1_seis_norm = ((test1_seismic - train_mean) / train_std).astype(np.float32)
test2_seis_norm = ((test2_seismic - train_mean) / train_std).astype(np.float32)

# --- Dataset'ler ---
# Train: inline + crossline ayni anda
train_inline_ds = F3Dataset25D(
    train_seis_norm, train_labels, train_inline_idx,
    axis=0, transform=train_transform, augment_polarity=True)

train_xline_ds = F3Dataset25D(
    train_seis_norm, train_labels, train_xline_idx,
    axis=1, transform=train_transform, augment_polarity=True)

train_ds = ConcatDataset([train_inline_ds, train_xline_ds])

# Val: sadece blok-bazli ayrilmis inline'lar (TTA'siz, augmentasyonsuz)
val_ds = F3Dataset25D(train_seis_norm, train_labels, val_inline_idx, axis=0)

# Test: zaten ayri volume (axis=0 cunku npy zaten 2D slice olarak sakli)
test1_ds = F3Dataset25D(test1_seis_norm, test1_labels, np.arange(test1_seismic.shape[0]), axis=0)
test2_ds = F3Dataset25D(test2_seis_norm, test2_labels, np.arange(test2_seismic.shape[0]), axis=0)

print(f"Train inline : {len(train_inline_ds)} slice")
print(f"Train xline  : {len(train_xline_ds)} slice")
print(f"Train toplam : {len(train_ds)} slice")
print(f"Val          : {len(val_ds)} slice")
print(f"Test1        : {len(test1_ds)} slice")
print(f"Test2        : {len(test2_ds)} slice")

In [ ]:
# --- WeightedRandomSampler: nadir sinifli slice'lara daha fazla agirlik ---
RARE_CLASSES = [4, 5]  # Zechstein, Under Zech
RARE_BOOST   = 10.0

def compute_slice_weights(labels, indices, axis, rare_classes=RARE_CLASSES, boost=RARE_BOOST):
    weights = []
    for i in indices:
        sl = labels[i] if axis == 0 else labels[:, i]
        total_px = sl.size
        rare_px  = sum(int((sl == c).sum()) for c in rare_classes)
        w = 1.0 + (rare_px / total_px) * boost
        weights.append(w)
    return weights

w_inline = compute_slice_weights(train_labels, train_inline_idx, axis=0)
w_xline  = compute_slice_weights(train_labels, train_xline_idx, axis=1)
all_weights = w_inline + w_xline  # ConcatDataset sirasina uygun

sampler = WeightedRandomSampler(
    weights=all_weights,
    num_samples=len(train_ds),
    replacement=True
)

# Agirlik istatistikleri
w_arr = np.array(all_weights)
print(f"Sampling agirliklari: min={w_arr.min():.2f}  max={w_arr.max():.2f}  "
      f"mean={w_arr.mean():.2f}  median={np.median(w_arr):.2f}")
print(f"Agirlik > 1.5 olan slice'lar: {(w_arr > 1.5).sum()} / {len(w_arr)}  "
      f"({(w_arr > 1.5).mean()*100:.1f}%)")

# --- DataLoader'lar ---
NUM_WORKERS = 0 if sys.platform == "win32" else 2
pin = device.type == "cuda"

train_loader = DataLoader(train_ds,  batch_size=BATCH_SIZE, sampler=sampler,
                          num_workers=NUM_WORKERS, pin_memory=pin)
val_loader   = DataLoader(val_ds,    batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=pin)
test1_loader = DataLoader(test1_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
test2_loader = DataLoader(test2_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

imgs, masks = next(iter(train_loader))
print(f"\nBatch — goruntu: {imgs.shape}  maske: {masks.shape}")
print(f"  -> 3 kanal = [slice-1, slice, slice+1] (2.5D)")
print(f"Efektif batch size: {BATCH_SIZE} x {ACCUM_STEPS} = {BATCH_SIZE * ACCUM_STEPS}")

## 5. Model — DeepLabV3+ (ResNet-50, in_channels=3)
3 kanallı input → ImageNet pretrained ağırlıklar doğrudan kullanılabilir.

In [ ]:
model = smp.DeepLabV3Plus(
    encoder_name="resnet50",
    encoder_weights="imagenet",
    in_channels=3,
    classes=NUM_CLASSES,
    activation=None,
    encoder_output_stride=16,
    decoder_atrous_rates=(6, 12, 18),
).to(device)

n_params = sum(p.numel() for p in model.parameters())
print(f"Toplam parametre: {n_params:,}")
print(f"in_channels=3 — ImageNet pretrained agirliklar dogrudan uyumlu!")

with torch.no_grad():
    dummy = torch.randn(2, 3, 256, 256).to(device)
    out = model(dummy)
    print(f"Cikti boyutu: {out.shape}  (beklenen: [2, {NUM_CLASSES}, 256, 256])")
del dummy, out
if device.type == 'cuda':
    torch.cuda.empty_cache()
print("Model hazir.")

## 6. Loss (Focal + Dice), AdamW, OneCycleLR

In [ ]:
class FocalLoss(nn.Module):
    def __init__(self, alpha, gamma=2.0):
        super().__init__()
        self.register_buffer('alpha', alpha)
        self.gamma = gamma

    def forward(self, pred, target):
        ce = F.cross_entropy(pred, target, reduction='none')
        pt = torch.exp(-ce)
        alpha_t = self.alpha[target]
        loss = alpha_t * ((1 - pt) ** self.gamma) * ce
        return loss.mean()


class DiceLoss(nn.Module):
    def __init__(self, n_classes=NUM_CLASSES, smooth=1.0):
        super().__init__()
        self.n_classes = n_classes; self.smooth = smooth

    def forward(self, pred, target):
        pred_soft = torch.softmax(pred, dim=1)
        target_oh = F.one_hot(target, self.n_classes).permute(0, 3, 1, 2).float()
        dice = 0.0
        for c in range(self.n_classes):
            p = pred_soft[:, c]; t = target_oh[:, c]
            inter = (p * t).sum()
            dice += (2 * inter + self.smooth) / (p.sum() + t.sum() + self.smooth)
        return 1 - dice / self.n_classes


class CombinedLossV5(nn.Module):
    def __init__(self, alpha, gamma=2.0):
        super().__init__()
        self.focal = FocalLoss(alpha, gamma)
        self.dice  = DiceLoss()

    def forward(self, pred, target):
        return 0.5 * self.focal(pred, target) + 0.5 * self.dice(pred, target)


NUM_EPOCHS = 100
criterion  = CombinedLossV5(focal_alpha_t, gamma=2.0).to(device)
optimizer  = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-3)
scheduler  = torch.optim.lr_scheduler.OneCycleLR(
    optimizer,
    max_lr=3e-4,
    epochs=NUM_EPOCHS,
    steps_per_epoch=len(train_loader) // ACCUM_STEPS + 1,
    pct_start=0.1,
    anneal_strategy='cos',
    div_factor=10,
    final_div_factor=100,
)

print("Loss  : 0.5 * Focal(gamma=2) + 0.5 * Dice")
print("Optim : AdamW  lr=3e-4  wd=1e-3")
print("Sched : OneCycleLR  max_lr=3e-4  warmup=10%")
print(f"Epoch : {NUM_EPOCHS}")
print(f"Grad Accumulation: {ACCUM_STEPS} steps")

## 7. Metrik Fonksiyonları

In [ ]:
def compute_metrics(preds_all, targets_all, n=NUM_CLASSES):
    res = {}
    res['PA'] = float((preds_all == targets_all).sum() / len(targets_all))
    cm = confusion_matrix(targets_all, preds_all, labels=list(range(n)))

    row_sums  = cm.sum(axis=1).astype(float)
    class_acc = np.where(row_sums > 0, cm.diagonal() / row_sums, 0.0)
    res['MCA'] = float(class_acc.mean())
    res['per_class_acc'] = class_acc.tolist()

    ious = []
    for c in range(n):
        inter = cm[c, c]; union = cm[c,:].sum() + cm[:,c].sum() - inter
        ious.append(float(inter / (union + 1e-8)))
    res['mIoU'] = float(np.mean(ious))
    res['per_class_iou'] = ious

    dices = []
    for c in range(n):
        tp = cm[c,c]; fp = cm[:,c].sum()-tp; fn = cm[c,:].sum()-tp
        dices.append(float(2*tp / (2*tp + fp + fn + 1e-8)))
    res['mean_dice'] = float(np.mean(dices))
    res['per_class_dice'] = dices
    res['confusion_matrix'] = cm.tolist()
    return res


def print_metrics(metrics, title='Model'):
    print(f'\n{"="*60}')
    print(f' {title}')
    print(f'{"="*60}')
    print(f'  PA        : {metrics["PA"]*100:.2f}%')
    print(f'  MCA       : {metrics["MCA"]*100:.2f}%')
    print(f'  mIoU      : {metrics["mIoU"]*100:.2f}%')
    print(f'  Mean Dice : {metrics["mean_dice"]*100:.2f}%')
    print('  Per-class IoU:')
    for i, name in enumerate(CLASS_NAMES):
        iou  = metrics['per_class_iou'][i]
        dice = metrics['per_class_dice'][i]
        print(f"    S{i} {name:20s}: IoU={iou:.4f}  Dice={dice:.4f}")


print("Metrik fonksiyonlari hazir.")

## 8. Eğitim (100 Epoch + Grad Accum + Early Stop)
> Checkpoint otomatik kaydedilir — kaldığı yerden devam eder.

In [ ]:
CHECKPOINT_PATH = CHECKPOINTS / "deeplabv3plus_v5_checkpoint.pth"
BEST_MODEL_PATH = CHECKPOINTS / "deeplabv3plus_v5_best.pth"

use_amp = device.type == "cuda"  # MPS henuz AMP desteklemiyor
scaler  = torch.amp.GradScaler("cuda") if use_amp else None

start_epoch   = 0
best_val_miou = 0.0
patience_counter = 0
PATIENCE = 25
history = {'train_loss': [], 'val_loss': [], 'val_miou': [], 'val_dice': [], 'lr': []}

if CHECKPOINT_PATH.exists():
    print("Checkpoint bulundu, devam ediliyor...")
    ckpt = torch.load(CHECKPOINT_PATH, map_location=device, weights_only=False)
    model.load_state_dict(ckpt['model_state'])
    optimizer.load_state_dict(ckpt['optimizer_state'])
    scheduler.load_state_dict(ckpt['scheduler_state'])
    start_epoch   = ckpt['epoch'] + 1
    best_val_miou = ckpt['best_val_miou']
    history       = ckpt['history']
    patience_counter = ckpt.get('patience_counter', 0)
    if use_amp and 'scaler_state' in ckpt:
        scaler.load_state_dict(ckpt['scaler_state'])
    print(f"Epoch {start_epoch}/{NUM_EPOCHS} — onceki en iyi mIoU: {best_val_miou:.4f}")
else:
    print("Sifirdan basliyor...")

print(f"\nCihaz: {device}  |  AMP: {use_amp}  |  Kalan epoch: {NUM_EPOCHS - start_epoch}")
print(f"Grad Accum: {ACCUM_STEPS}  |  Early Stop Patience: {PATIENCE}")
print(f"Train: {len(train_ds)} slice (inline+xline)  |  Val: {len(val_ds)} slice (blok-bazli)")
print("=" * 80)

for epoch in range(start_epoch, NUM_EPOCHS):
    t0 = time.time()

    # --- TRAIN ---
    model.train()
    optimizer.zero_grad()
    train_loss = 0.0
    for step, (imgs, masks) in enumerate(train_loader):
        imgs, masks = imgs.to(device, non_blocking=True), masks.to(device, non_blocking=True)
        if use_amp:
            with torch.amp.autocast("cuda"):
                preds = model(imgs)
                loss  = criterion(preds, masks) / ACCUM_STEPS
            scaler.scale(loss).backward()
        else:
            preds = model(imgs)
            loss  = criterion(preds, masks) / ACCUM_STEPS
            loss.backward()

        if (step + 1) % ACCUM_STEPS == 0 or (step + 1) == len(train_loader):
            if use_amp:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                scaler.step(optimizer)
                scaler.update()
            else:
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()
            optimizer.zero_grad()
            scheduler.step()

        train_loss += loss.item() * ACCUM_STEPS
    train_loss /= len(train_loader)

    # --- VALIDATION ---
    model.eval()
    val_loss = 0.0
    all_p, all_t = [], []
    with torch.no_grad():
        for imgs, masks in val_loader:
            imgs, masks = imgs.to(device, non_blocking=True), masks.to(device, non_blocking=True)
            if use_amp:
                with torch.amp.autocast("cuda"):
                    preds = model(imgs)
            else:
                preds = model(imgs)
            val_loss += criterion(preds, masks).item()
            all_p.append(preds.argmax(1).cpu().numpy().flatten())
            all_t.append(masks.cpu().numpy().flatten())
    val_loss   /= len(val_loader)
    val_metrics = compute_metrics(np.concatenate(all_p), np.concatenate(all_t))

    lr = optimizer.param_groups[0]['lr']
    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['val_miou'].append(val_metrics['mIoU'])
    history['val_dice'].append(val_metrics['mean_dice'])
    history['lr'].append(lr)

    marker = ""
    if val_metrics['mIoU'] > best_val_miou:
        best_val_miou = val_metrics['mIoU']
        torch.save(model.state_dict(), BEST_MODEL_PATH)
        marker = "  *** BEST ***"
        patience_counter = 0
    else:
        patience_counter += 1

    ckpt_data = {
        'epoch': epoch, 'model_state': model.state_dict(),
        'optimizer_state': optimizer.state_dict(),
        'scheduler_state': scheduler.state_dict(),
        'best_val_miou': best_val_miou, 'history': history,
        'patience_counter': patience_counter,
    }
    if use_amp:
        ckpt_data['scaler_state'] = scaler.state_dict()
    torch.save(ckpt_data, CHECKPOINT_PATH)

    miou  = val_metrics['mIoU']
    mdice = val_metrics['mean_dice']
    print(f"Ep {epoch+1:03d}/{NUM_EPOCHS} | Train: {train_loss:.4f} | Val: {val_loss:.4f} | "
          f"mIoU: {miou:.4f} | Dice: {mdice:.4f} | lr: {lr:.2e} | {time.time()-t0:.0f}s{marker}")

    if patience_counter >= PATIENCE:
        print(f"\nEarly stopping — {PATIENCE} epoch boyunca iyilesme yok.")
        break

print(f"\nEgitim tamamlandi! En iyi val mIoU: {best_val_miou:.4f}")

## 9. Test Değerlendirmesi + TTA (HFlip + Polarity)

In [ ]:
model.load_state_dict(torch.load(BEST_MODEL_PATH, map_location=device, weights_only=True))
model.eval()
print(f"En iyi model yuklendi: {BEST_MODEL_PATH}")


def run_eval_tta(loader, tta=True):
    all_p, all_t = [], []
    with torch.no_grad():
        for imgs, masks in loader:
            imgs_dev = imgs.to(device, non_blocking=True)

            if use_amp:
                with torch.amp.autocast("cuda"):
                    logits = model(imgs_dev)
            else:
                logits = model(imgs_dev)
            probs = torch.softmax(logits, dim=1)

            if tta:
                # HFlip
                imgs_hf = torch.flip(imgs_dev, dims=[3])
                if use_amp:
                    with torch.amp.autocast("cuda"):
                        logits_hf = model(imgs_hf)
                else:
                    logits_hf = model(imgs_hf)
                probs += torch.softmax(torch.flip(logits_hf, dims=[3]), dim=1)

                # Polarity inversion
                imgs_neg = -imgs_dev
                if use_amp:
                    with torch.amp.autocast("cuda"):
                        logits_neg = model(imgs_neg)
                else:
                    logits_neg = model(imgs_neg)
                probs += torch.softmax(logits_neg, dim=1)

                probs /= 3.0

            all_p.append(probs.argmax(1).cpu().numpy().flatten())
            all_t.append(masks.numpy().flatten())
    return np.concatenate(all_p), np.concatenate(all_t)


# TTA'siz
print("--- TTA'siz ---")
p1_no, t1_no = run_eval_tta(test1_loader, tta=False)
p2_no, t2_no = run_eval_tta(test2_loader, tta=False)
m1_no = compute_metrics(p1_no, t1_no)
m2_no = compute_metrics(p2_no, t2_no)
print(f"  Test1 mIoU: {m1_no['mIoU']*100:.2f}%  |  Test2 mIoU: {m2_no['mIoU']*100:.2f}%")

# TTA'li
print("\n--- TTA ile ---")
p1, t1 = run_eval_tta(test1_loader, tta=True)
p2, t2 = run_eval_tta(test2_loader, tta=True)
metrics1    = compute_metrics(p1, t1)
metrics2    = compute_metrics(p2, t2)
metrics_all = compute_metrics(np.concatenate([p1,p2]), np.concatenate([t1,t2]))

print_metrics(metrics1,    "Test1 (Inline) — TTA")
print_metrics(metrics2,    "Test2 (Crossline — Genelleme) — TTA")
print_metrics(metrics_all, "Birlesik Test1 + Test2 — TTA")

results = {
    "model": "DeepLabV3+ v5 (ResNet-50, 2.5D, Multi-View, Focal+Dice, TTA)",
    "test1": metrics1, "test2": metrics2, "combined": metrics_all, "history": history
}
with open(METRICS_DIR / "deeplabv3plus_v5_metrics.json", "w") as f:
    json.dump(results, f, indent=2)
print(f"\nMetrikler kaydedildi: {METRICS_DIR}/deeplabv3plus_v5_metrics.json")

## 10. Eğitim Eğrileri

In [ ]:
epochs_x = range(1, len(history['train_loss']) + 1)
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

axes[0].plot(epochs_x, history['train_loss'], label='Train', color='steelblue')
axes[0].plot(epochs_x, history['val_loss'],   label='Val',   color='coral')
axes[0].set_title('Loss'); axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(epochs_x, history['val_miou'], label='mIoU',  color='green')
axes[1].plot(epochs_x, history['val_dice'], label='Dice',  color='purple')
axes[1].set_title('Metrikler'); axes[1].legend(); axes[1].grid(alpha=0.3)

axes[2].plot(epochs_x, history['lr'], color='darkorange')
axes[2].set_title('Learning Rate'); axes[2].grid(alpha=0.3)

for ax in axes: ax.set_xlabel('Epoch')
plt.suptitle("DeepLabV3+ v5 — Egitim Sureci (2.5D Multi-View)", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "training_curves_v5.png", dpi=150, bbox_inches="tight")
plt.show()

## 11. Segmentasyon Karşılaştırması (Test1 + Test2)

In [ ]:
model.eval()

fig, axes = plt.subplots(6, 3, figsize=(15, 24))

# 3 inline (test1) + 3 crossline (test2)
test1_sample = np.linspace(0, len(test1_ds)-1, 3, dtype=int)
test2_sample = np.linspace(0, len(test2_ds)-1, 3, dtype=int)

with torch.no_grad():
    for row, (ds, idx, label) in enumerate(
        [(test1_ds, i, f"Inline #{i}") for i in test1_sample] +
        [(test2_ds, i, f"Crossline #{i}") for i in test2_sample]
    ):
        img, mask = ds[idx]
        inp = img.unsqueeze(0).to(device)
        if use_amp:
            with torch.amp.autocast("cuda"):
                pred = model(inp).argmax(1).squeeze().cpu().numpy()
        else:
            pred = model(inp).argmax(1).squeeze().cpu().numpy()

        # Ortadaki kanal (asil slice)
        axes[row, 0].imshow(img[1].numpy(), cmap="seismic", vmin=-2, vmax=2, aspect="auto")
        axes[row, 0].set_title(f"Sismik — {label}", fontsize=9); axes[row, 0].axis("off")

        axes[row, 1].imshow(mask.numpy(), cmap=cmap_facies, vmin=-0.5, vmax=5.5, aspect="auto")
        axes[row, 1].set_title("Gercek Fasiyes", fontsize=9); axes[row, 1].axis("off")

        axes[row, 2].imshow(pred, cmap=cmap_facies, vmin=-0.5, vmax=5.5, aspect="auto")
        axes[row, 2].set_title("v5 Tahmini", fontsize=9); axes[row, 2].axis("off")

patches_vis = [mpatches.Patch(color=FACIES_COLORS[j], label=f"S{j}: {CLASS_NAMES[j]}") for j in range(NUM_CLASSES)]
fig.legend(handles=patches_vis, loc="lower center", ncol=3, fontsize=9, bbox_to_anchor=(0.5, -0.01))
plt.suptitle("DeepLabV3+ v5 — Inline + Crossline Segmentasyon", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "segmentation_comparison_v5.png", dpi=150, bbox_inches="tight")
plt.show()

## 12. Confusion Matrix

In [ ]:
cm_arr  = np.array(metrics_all['confusion_matrix'])
cm_norm = cm_arr.astype(float) / cm_arr.sum(axis=1, keepdims=True).clip(min=1)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
for ax, data, title in zip(axes, [cm_arr, cm_norm], ["Ham Sayilar", "Normalize (satir %)"]):
    im = ax.imshow(data, cmap="Blues")
    ax.set_xticks(range(NUM_CLASSES)); ax.set_yticks(range(NUM_CLASSES))
    ax.set_xticklabels([f"S{j}\n{CLASS_NAMES[j]}" for j in range(NUM_CLASSES)], fontsize=8, rotation=15)
    ax.set_yticklabels([f"S{j} {CLASS_NAMES[j]}" for j in range(NUM_CLASSES)], fontsize=8)
    ax.set_xlabel("Tahmin"); ax.set_ylabel("Gercek")
    ax.set_title(f"Confusion Matrix — {title}", fontsize=11)
    plt.colorbar(im, ax=ax)
    for i in range(NUM_CLASSES):
        for j in range(NUM_CLASSES):
            val = f"{data[i,j]:.2f}" if title != "Ham Sayilar" else f"{int(data[i,j]):,}"
            ax.text(j, i, val, ha="center", va="center", fontsize=6,
                    color="white" if data[i,j] > data.max()*0.5 else "black")

plt.suptitle("DeepLabV3+ v5 — Confusion Matrix", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "confusion_matrix_v5.png", dpi=150, bbox_inches="tight")
plt.show()

## 13. Per-class IoU / Dice

In [ ]:
iou_vals  = metrics_all['per_class_iou']
dice_vals = metrics_all['per_class_dice']
x = np.arange(NUM_CLASSES); width = 0.35

fig, ax = plt.subplots(figsize=(11, 5))
b1 = ax.bar(x - width/2, iou_vals,  width, label="IoU",  color=FACIES_COLORS, alpha=0.85, edgecolor="k", lw=0.5)
b2 = ax.bar(x + width/2, dice_vals, width, label="Dice", color=FACIES_COLORS, alpha=0.55, edgecolor="k", lw=0.5, hatch="///")

for bar, v in zip(b1, iou_vals):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01, f"{v:.3f}", ha="center", fontsize=8)
for bar, v in zip(b2, dice_vals):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01, f"{v:.3f}", ha="center", fontsize=8)

miou_val  = metrics_all['mIoU']
mdice_val = metrics_all['mean_dice']
ax.axhline(miou_val,  color="green",  ls="--", lw=1.2, label=f"mIoU={miou_val:.3f}")
ax.axhline(mdice_val, color="purple", ls="--", lw=1.2, label=f"mDice={mdice_val:.3f}")

ax.set_xticks(x)
ax.set_xticklabels([f"S{j}\n{CLASS_NAMES[j]}" for j in range(NUM_CLASSES)], fontsize=9)
ax.set_ylabel("Skor"); ax.set_ylim(0, 1.1)
ax.set_title("Per-class IoU ve Dice (Birlenik Test) — v5", fontsize=11)
ax.legend(fontsize=9); ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig(FIGURES_DIR / "per_class_metrics_v5.png", dpi=150, bbox_inches="tight")
plt.show()

## 14. v3 vs v5 Karşılaştırma

In [ ]:
v3_metrics_path = PROJECT_DIR / "deeplabv3plus_metrics.json"
if v3_metrics_path.exists():
    with open(v3_metrics_path) as f:
        v3 = json.load(f)

    print("=" * 70)
    print(" v3 vs v5 KARSILASTIRMA")
    print("=" * 70)
    print(f"  {'Metrik':<25} {'v3':>10} {'v5':>10} {'Fark':>10}")
    print("-" * 70)

    comparisons = [
        ("Test1 mIoU",     v3['test1']['mIoU'],      metrics1['mIoU']),
        ("Test1 Dice",     v3['test1']['mean_dice'],  metrics1['mean_dice']),
        ("Test1 PA",       v3['test1']['PA'],         metrics1['PA']),
        ("Test2 mIoU",     v3['test2']['mIoU'],       metrics2['mIoU']),
        ("Test2 Dice",     v3['test2']['mean_dice'],   metrics2['mean_dice']),
        ("Test2 PA",       v3['test2']['PA'],          metrics2['PA']),
        ("Combined mIoU",  v3['combined']['mIoU'],     metrics_all['mIoU']),
        ("Combined Dice",  v3['combined']['mean_dice'], metrics_all['mean_dice']),
        ("Combined PA",    v3['combined']['PA'],       metrics_all['PA']),
    ]

    for name, old, new in comparisons:
        diff = (new - old) * 100
        arrow = "+" if diff > 0 else "-" if diff < 0 else "="
        print(f"  {name:<23} {old*100:>9.2f}% {new*100:>9.2f}% {arrow}{abs(diff):>8.2f}pp")

    print(f"\n  Per-class Test2 IoU (en kritik — genelleme testi):")
    for i, name in enumerate(CLASS_NAMES):
        old_iou = v3['test2']['per_class_iou'][i]
        new_iou = metrics2['per_class_iou'][i]
        diff = (new_iou - old_iou) * 100
        arrow = "+" if diff > 0 else "-"
        print(f"    S{i} {name:20s}: {old_iou*100:>6.2f}% -> {new_iou*100:>6.2f}%  ({arrow}{abs(diff):.2f}pp)")
else:
    print("v3 metrik dosyasi bulunamadi.")

print(f"\n{'=' * 70}")
print(" v5 FINAL SONUCLAR")
print(f"{'=' * 70}")
print(f"  Test1 mIoU   : {metrics1['mIoU']*100:.2f}%  (inline)")
print(f"  Test2 mIoU   : {metrics2['mIoU']*100:.2f}%  (crossline — genelleme)")
print(f"  Combined mIoU: {metrics_all['mIoU']*100:.2f}%")
print(f"\n  Degisiklikler:")
print(f"    2.5D input (3ch)          : komsu slice sürekliligi")
print(f"    Inline + Crossline egitim : multi-view genelleme")
print(f"    Blok-bazli val split      : durust degerlendirme")
print(f"    WeightedRandomSampler     : nadir sinif iyilestirme")
print(f"    VerticalFlip kaldirildi   : fiziksel tutarlilik")
print(f"\n  Model : {BEST_MODEL_PATH}")
print(f"  JSON  : {METRICS_DIR}/deeplabv3plus_v5_metrics.json")